In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="fact_financial_pnl",
    choices=["discount", "sales", "fact_financial_pnl", "fact_financial_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.gl_report limit 1

In [0]:
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df(df):

    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")

    # Step 1: Joined both master tables upfront
    df_all_masters = df.join(
        df_coa_master, 
        df_coa_master["account_number"].cast("string") == df["accountNumber"], 
        'inner'
    ).join(
        df_location_master,
        col("location") == df_location_master.netsuite_location_name,
        'left'
    )

    # Step 2: Created detail rows with all columns
    df_detail = df_all_masters.select(
        col("location"),
        col("month"),
        col("year"),
        col("account_name"),
        col("name"),
        col("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("amount"),
        # col("debit"),
        # col("credit"),
        # col("finalamount"),
        # Location master columns
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("store_type"),
        col("city"),
        lit("Detail").alias("Detail/Total")
    )

    # Step 3: Created aggregations with location master columns in groupBy
    df_subgroup_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group", "sub_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        sum("amount").alias("amount"),
        # sum("debit").alias("debit"),
        # sum("credit").alias("credit"),
        # sum("finalamount").alias("finalamount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("sub_group")).alias("account_name"),
        concat(lit("Total "), col("sub_group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("amount"),
        # col("debit"),
        # col("credit"),
        # col("finalamount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        lit("Total").alias("Detail/Total")
    )

    df_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        sum("amount").alias("amount"),
        # sum("debit").alias("debit"),
        # sum("credit").alias("credit"),
        # sum("finalamount").alias("finalamount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("group")).alias("account_name"),
        concat(lit("Total "), col("group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        col("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        # col("debit"),
        # col("credit"),
        # col("finalamount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        lit("Total").alias("Detail/Total")
    )

    df_majour_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        sum("amount").alias("amount"),
        # sum("debit").alias("debit"),
        # sum("credit").alias("credit"),
        # sum("finalamount").alias("finalamount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        concat(lit("Total "), col("majour_group")).alias("account_name"),
        concat(lit("Total "), col("majour_group")).alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        col("majour_group"),
        lit("N/A").alias("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        # col("debit"),
        # col("credit"),
        # col("finalamount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        lit("Grand Total").alias("Detail/Total")
    )

    # Gross Profit = Total Sales - Total Purchases (sum_order 4)
    df_gross_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        (sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        lit("Gross Profit").alias("account_name"),
        lit("Gross Profit").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Gross Profit").alias("majour_group"),
        lit("N/A").alias("group"),
        lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        lit("Grand Total").alias("Detail/Total")
    )
    
    # Operating Profit = Gross Profit - Total Overheads (sum_order 4)
    df_operating_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        ((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("Operating Profit").alias("account_name"),
        lit("Operating Profit").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Operating Profit").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("parent_company"),
        col("country_code"),
        col('zone'), col("city"),
        lit("Grand Total").alias("Detail/Total")
    )

    # EBITDA = Operating Profit + Total Depreciation & Amortization 
    df_ebitda = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) +
        sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("EBITDA").alias("account_name"),
        lit("EBITDA").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("EBITDA").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("parent_company"),
        col("country_code"),
        col('zone'), col("city"),
        lit("Grand Total").alias("Detail/Total")
    )

    # Net Profit = Operating Profit - Total Finance Costs - Total Tax (sum_order 4)
    df_net_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company","country_code","zone","store_type", "city"
    ).agg(
        (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
        sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) -
        sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0)) -
        sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0))).alias("amount")
    ).select(
        col("location"), col("month"), col("year"),
        lit("Net Profit / (Loss)").alias("account_name"),
        lit("Net Profit / (Loss)").alias("name"),
        lit(None).cast("string").alias("accoun_type"),
        lit("Net Profit / (Loss)").alias("majour_group"),
        lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
        col("amount"),
        col("netsuite_location_name"), col("type"), col("location_id"),
        col("brand_id"), col("company_id"), col("store_type"), col("parent_company"),
        col("country_code"),
        col('zone'), col("city"),
        lit("Grand Total").alias("Detail/Total")
    )

    # Step 4: Update union to include all calculated rows
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))

    # Step 5: Extracted year and prepare for PY calculations
    df_current = df_combined.withColumn("year", year(col("year")))

    # Step 6: Calculated Previous Year Sales (PY) - Self join
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("location").alias("py_location"),
        col("account_name").alias("py_account_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )

    df_with_py = df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.location") == col("py_location")) &  
        (col("curr.account_name") == col("py_account_name")) &  
        (col("curr.month") == col("py_month")),  
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )

    # Step 7: Calculated Net Sales with window functions
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")

    df_with_calculations = df_with_py.withColumn(
        "Actual Net Sales",
        when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))
    ).withColumn(
        "PY Netsales",
        when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))
    ).withColumn(
        "Location Actual Net Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "Location PY Net Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "Brand Act Nets Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "Brand PY NetSales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "Company Act Nets Sales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_company)
    ).withColumn(
        "Company PY NetSales",
        sum(when(
            (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_company)
    )

    # Step 8: Create final output with all required columns
    df_final = df_with_calculations.select(
        col("account_name"),
        col("name"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        # lit(None).cast("string"),
        col("accoun_type"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        # lit(None).cast("string").alias("Cluster"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        col("amount"),
        col("year"),
        col("month"),
        col("Actual Net Sales"),
        col("PY Netsales"),
        col("Brand Act Nets Sales"),
        col("Brand PY NetSales"),
        col("Company Act Nets Sales"),
        col("Company PY NetSales"),
        col("Detail/Total")
    )

    
    fact_financial_pnl = to_snake_case_df(df_final)
    return fact_financial_pnl

In [0]:
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df


def enrich_df2(df):

    df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
    df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")

    # Step 1: Join both master tables upfront
    df_all_masters = df.join(
        df_coa_master, 
        df_coa_master["account_number"].cast("string") == df["accountNumber"], 
        'inner'
    ).join(
        df_location_master,
        col("location") == df_location_master.netsuite_location_name,
        'left'
    )

    # Step 2: Create detail rows - CORRECTED ORDER
    df_detail = df_all_masters.select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        col("account_name"),            # 4
        col("name"),                    # 5
        col("accoun_type"),             # 6
        col("majour_group"),            # 7
        col("group"),                   # 8
        col("sub_group"),               # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Detail").alias("Detail/Total")  # 21
    )

    # Step 3: Sub group totals - SAME ORDER
    df_subgroup_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group", "sub_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("sub_group")).alias("account_name"),  # 4
        concat(lit("Total "), col("sub_group")).alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),                   # 8
        lit(None).alias("sub_group"),               # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Total").alias("Detail/Total")  # 21
    )

    # Group totals - SAME ORDER
    df_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group", "group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("group")).alias("account_name"),      # 4
        concat(lit("Total "), col("group")).alias("name"),              # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),                   # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Total").alias("Detail/Total")  # 21
    )

    # Major group totals - SAME ORDER
    df_majour_group_totals = df_all_masters.groupBy(
        "location", "month", "year", "majour_group",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        concat(lit("Total "), col("majour_group")).alias("account_name"),  # 4
        concat(lit("Total "), col("majour_group")).alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Gross Profit - SAME ORDER
    df_gross_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
     abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Gross Profit").alias("account_name"),  # 4
        lit("Gross Profit").alias("name"),          # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),  # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )
    
    # Operating Profit - SAME ORDER
    df_operating_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (abs((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
          abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
         abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Operating Profit").alias("account_name"),  # 4
        lit("Operating Profit").alias("name"),          # 5
        lit(None).alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),  # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # EBITDA - SAME ORDER
    df_ebitda = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (((abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
           abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
          abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) +
         abs(sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("EBITDA").alias("account_name"),            # 4
        lit("EBITDA").alias("name"),                    # 5
        lit(None).cast("string").alias("accoun_type"),  # 6
        lit(None).alias("majour_group"),            # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Net Profit - SAME ORDER
    df_net_profit = df_all_masters.groupBy(
        "location", "month", "year",
        "netsuite_location_name", "type", "location_id", "brand_id", 
        "company_id", "parent_company", "country_code", "zone", "store_type", "city"
    ).agg(
        (((abs(sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0))) - 
           abs(sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0)))) -
          abs(sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0)))) -
         abs(sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0))) -
         abs(sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0)))).alias("amount")
    ).select(
        col("location"),                # 1
        col("month"),                   # 2
        col("year"),                    # 3
        lit("Net Profit/(Loss)").alias("account_name"),   # 4
        lit("Net Profit/(Loss)").alias("name"),           # 5
        lit(None).alias("accoun_type"),      # 6
        lit(None).alias("majour_group"),   # 7
        lit(None).alias("group"),      # 8
        lit(None).alias("sub_group"),  # 9
        col("amount"),                  # 10
        col("netsuite_location_name"),  # 11
        col("type"),                    # 12
        col("location_id"),             # 13
        col("brand_id"),                # 14
        col("company_id"),              # 15
        col("parent_company"),          # 16
        col("country_code"),            # 17
        col("zone"),                    # 18
        col("store_type"),              # 19
        col("city"),                    # 20
        lit("Grand Total").alias("Detail/Total")  # 21
    )

    # Step 4: Union all dataframes
    df_combined = (df_detail
        .unionAll(df_subgroup_totals)
        .unionAll(df_group_totals)
        .unionAll(df_majour_group_totals)
        .unionAll(df_gross_profit)
        .unionAll(df_operating_profit)
        .unionAll(df_ebitda)
        .unionAll(df_net_profit))

    # Step 5: Extract year and prepare for PY calculations
    df_current = df_combined.withColumn("year", year(col("year")))

    # Step 6: Calculate Previous Year Sales (PY) - Self join
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("location").alias("py_location"),
        col("account_name").alias("py_account_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )

    df_with_py = df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.location") == col("py_location")) &  
        (col("curr.account_name") == col("py_account_name")) &  
        (col("curr.month") == col("py_month")),  
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )

    # Step 7: Calculate Net Sales with window functions
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")

    df_with_calculations = df_with_py.withColumn(
        "actual_net_sales",
        when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))
    ).withColumn(
        "py_net_sales",
        when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))
    ).withColumn(
        "location_actual_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "location_py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_location)
    ).withColumn(
        "brand_act_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "brand_py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_brand)
    ).withColumn(
        "company_act_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("amount")
        ).otherwise(lit(0.0))).over(window_company)
    ).withColumn(
        "company_py_net_sales",
        sum(when(
            (col("account_name").like("Total Sales")),
            col("py_amount")
        ).otherwise(lit(0.0))).over(window_company)
    )

    # Step 8: Create final output with all required columns
    df_final = df_with_calculations.select(
        col("account_name"),
        col("name"),
        col("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("parent_company"),
        col("country_code"),
        col("zone"),
        col("store_type"),
        col("city"),
        col("Detail/Total"),
        col("amount"),
        col("year"),
        col("month"),
        col("actual_net_sales"),
        col("py_net_sales"),
        col("location_actual_net_sales"),
        col("location_py_net_sales"),
        col("brand_act_net_sales"),
        col("brand_py_net_sales"),
        col("company_act_net_sales"),
        col("company_py_net_sales")
    )

    fact_financial_pnl = to_snake_case_df(df_final)
    return fact_financial_pnl

In [0]:
# df = spark.read.table('fq_dev_catalog.bronze.gl_report')https://fqadfstoragedev.blob.core.windows.net/staging

In [0]:

df = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/Wastage/ALBAIK/2025/May/wastage_20250501_20250531.json')
exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
display(exploded_df)



In [0]:
df_detail = df_all_masters.select(
        col("location"),
        col("month"),
        col("year"),
        col("account_name"),
        col("name"),
        col("accoun_type"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        col("amount"),
        # col("debit"),
        # col("credit"),
        # col("finalamount"),
        # Location master columns
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        col("store_type"),
        col("city"),
        lit("Detail").alias("Detail/Total"),
        col('sort_order'),
        col('calculation_type')
    )

df_detail.display()

# Step 3: Created aggregations with location master columns in groupBy
df_subgroup_totals = df_all_masters.groupBy(
    "location", "month", "year", "majour_group", "group", "sub_group",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city", "sort_order", "calculation_type"
).agg(
    sum("amount").alias("amount"),
    # sum("debit").alias("debit"),
    # sum("credit").alias("credit"),
    # sum("finalamount").alias("finalamount")
).select(
    col("location"),
    col("month"),
    col("year"),
    concat(lit("Total "), col("sub_group")).alias("account_name"),
    concat(lit("Total "), col("sub_group")).alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    col("majour_group"),
    col("group"),
    col("sub_group"),
    col("amount"),
    # col("debit"),
    # col("credit"),
    # col("finalamount"),
    col("netsuite_location_name"),
    col("type"),
    col("location_id"),
    col("brand_id"),
    col("company_id"),
    col("store_type"),
    col("city"),
    lit("Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

display(df_subgroup_totals)
'''
df_group_totals = df_all_masters.groupBy(
    "location", "month", "year", "majour_group", "group",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    sum("amount").alias("amount"),
    # sum("debit").alias("debit"),
    # sum("credit").alias("credit"),
    # sum("finalamount").alias("finalamount")
).select(
    col("location"),
    col("month"),
    col("year"),
    concat(lit("Total "), col("group")).alias("account_name"),
    concat(lit("Total "), col("group")).alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    col("majour_group"),
    col("group"),
    lit("N/A").alias("sub_group"),
    col("amount"),
    # col("debit"),
    # col("credit"),
    # col("finalamount"),
    col("netsuite_location_name"),
    col("type"),
    col("location_id"),
    col("brand_id"),
    col("company_id"),
    col("store_type"),
    col("city"),
    lit("Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

df_majour_group_totals = df_all_masters.groupBy(
    "location", "month", "year", "majour_group",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    sum("amount").alias("amount"),
    # sum("debit").alias("debit"),
    # sum("credit").alias("credit"),
    # sum("finalamount").alias("finalamount")
).select(
    col("location"),
    col("month"),
    col("year"),
    concat(lit("Total "), col("majour_group")).alias("account_name"),
    concat(lit("Total "), col("majour_group")).alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    col("majour_group"),
    lit("N/A").alias("group"),
    lit("N/A").alias("sub_group"),
    col("amount"),
    # col("debit"),
    # col("credit"),
    # col("finalamount"),
    col("netsuite_location_name"),
    col("type"),
    col("location_id"),
    col("brand_id"),
    col("company_id"),
    col("store_type"),
    col("city"),
    lit("Grand Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

# Gross Profit = Total Sales - Total Purchases (sum_order 4)
df_gross_profit = df_all_masters.groupBy(
    "location", "month", "year",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    (sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
    sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))).alias("amount")
).select(
    col("location"),
    col("month"),
    col("year"),
    lit("Gross Profit").alias("account_name"),
    lit("Gross Profit").alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    lit("Gross Profit").alias("majour_group"),
    lit("N/A").alias("group"),
    lit("N/A").alias("sub_group"),
    col("amount"),
    col("netsuite_location_name"),
    col("type"),
    col("location_id"),
    col("brand_id"),
    col("company_id"),
    col("store_type"),
    col("city"),
    lit("Grand Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

# Operating Profit = Gross Profit - Total Overheads (sum_order 4)
df_operating_profit = df_all_masters.groupBy(
    "location", "month", "year",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    ((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
    sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
    sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))).alias("amount")
).select(
    col("location"), col("month"), col("year"),
    lit("Operating Profit").alias("account_name"),
    lit("Operating Profit").alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    lit("Operating Profit").alias("majour_group"),
    lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
    col("amount"),
    col("netsuite_location_name"), col("type"), col("location_id"),
    col("brand_id"), col("company_id"), col("store_type"), col("city"),
    lit("Grand Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

# EBITDA = Operating Profit + Total Depreciation & Amortization 
df_ebitda = df_all_masters.groupBy(
    "location", "month", "year",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
    sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
    sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) +
    sum(when(col("majour_group") == "Depreciation & Amortization", col("amount")).otherwise(0))).alias("amount")
).select(
    col("location"), col("month"), col("year"),
    lit("EBITDA").alias("account_name"),
    lit("EBITDA").alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    lit("EBITDA").alias("majour_group"),
    lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
    col("amount"),
    col("netsuite_location_name"), col("type"), col("location_id"),
    col("brand_id"), col("company_id"), col("store_type"), col("city"),
    lit("Grand Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

# Net Profit = Operating Profit - Total Finance Costs - Total Tax (sum_order 4)
df_net_profit = df_all_masters.groupBy(
    "location", "month", "year",
    "netsuite_location_name", "type", "location_id", "brand_id", 
    "company_id", "store_type", "city"
).agg(
    (((sum(when(col("majour_group") == "Sales", col("amount")).otherwise(0)) - 
    sum(when(col("majour_group") == "Purchases", col("amount")).otherwise(0))) -
    sum(when(col("majour_group") == "Overheads", col("amount")).otherwise(0))) -
    sum(when(col("majour_group") == "Finance Costs", col("amount")).otherwise(0)) -
    sum(when(col("majour_group") == "Tax", col("amount")).otherwise(0))).alias("amount")
).select(
    col("location"), col("month"), col("year"),
    lit("Net Profit / (Loss)").alias("account_name"),
    lit("Net Profit / (Loss)").alias("name"),
    lit(None).cast("string").alias("accoun_type"),
    lit("Net Profit / (Loss)").alias("majour_group"),
    lit("N/A").alias("group"), lit("N/A").alias("sub_group"),
    col("amount"),
    col("netsuite_location_name"), col("type"), col("location_id"),
    col("brand_id"), col("company_id"), col("store_type"), col("city"),
    lit("Grand Total").alias("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

# Step 4: Update union to include all calculated rows
df_combined = (df_detail
    .unionAll(df_subgroup_totals)
    .unionAll(df_group_totals)
    .unionAll(df_majour_group_totals)
    .unionAll(df_gross_profit)
    .unionAll(df_operating_profit)
    .unionAll(df_ebitda)
    .unionAll(df_net_profit))

# Step 5: Extracted year and prepare for PY calculations
df_current = df_combined.withColumn("year", year(col("year")))

# Step 6: Calculated Previous Year Sales (PY) - Self join
df_py = df_current.alias("py").select(
    (col("year") + 1).alias("year_join"),
    col("location").alias("py_location"),
    col("account_name").alias("py_account_name"),
    col("amount").alias("py_amount"),
    col("month").alias("py_month")
)

df_with_py = df_current.alias("curr").join(
    df_py,
    (col("curr.year") == col("year_join")) &  
    (col("curr.location") == col("py_location")) &  
    (col("curr.account_name") == col("py_account_name")) &  
    (col("curr.month") == col("py_month")),  
    "left"
).select(
    col("curr.*"),
    coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
)

# Step 7: Calculated Net Sales with window functions
window_location = Window.partitionBy("location", "year", "month")
window_brand = Window.partitionBy("brand_id", "year", "month")
window_company = Window.partitionBy("company_id", "year", "month")

df_with_calculations = df_with_py.withColumn(
    "Actual Net Sales",
    when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("amount")
    ).otherwise(lit(0.0))
).withColumn(
    "PY Netsales",
    when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("py_amount")
    ).otherwise(lit(0.0))
).withColumn(
    "Location Actual Net Sales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("amount")
    ).otherwise(lit(0.0))).over(window_location)
).withColumn(
    "Location PY Net Sales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("py_amount")
    ).otherwise(lit(0.0))).over(window_location)
).withColumn(
    "Brand Act Nets Sales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("amount")
    ).otherwise(lit(0.0))).over(window_brand)
).withColumn(
    "Brand PY NetSales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("py_amount")
    ).otherwise(lit(0.0))).over(window_brand)
).withColumn(
    "Company Act Nets Sales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("amount")
    ).otherwise(lit(0.0))).over(window_company)
).withColumn(
    "Company PY NetSales",
    sum(when(
        (col("majour_group") == "Sales") & (col("account_name").like("Total Sales")),
        col("py_amount")
    ).otherwise(lit(0.0))).over(window_company)
)

# Step 8: Create final output with all required columns
df_final = df_with_calculations.select(
    col("account_name"),
    col("name"),
    col("majour_group").alias("Major Group"),
    col("group").alias("Group"),
    col("sub_group").alias("sub_group"),
    lit(None).cast("string").alias("Alternate Group"),
    col("accoun_type").alias("accounttype"),
    col("netsuite_location_name").alias("Location"),
    col("type").alias("Location type (HO/Store)"),
    col("location_id").alias("Location Code"),
    col("brand_id").alias("Brand"),
    col("company_id").alias("Company"),
    # lit(None).cast("string").alias("Cluster"),
    col("store_type").alias("Location Mode (Mall/Drive thru/Stand alone)"),
    col("city").alias("Emirates"),
    col("amount").alias("Actual Value"),
    col("year").alias("Year"),
    col("month").alias("Month"),
    col("Actual Net Sales"),
    col("PY Netsales"),
    col("Brand Act Nets Sales"),
    col("Brand PY NetSales"),
    col("Company Act Nets Sales"),
    col("Company PY NetSales"),
    col("Detail/Total"),
    col('sort_order'),
    col('calculation_type')
)

df_final = df_final.orderBy('year', 'month', 'location_id', 'brand_id', 'company_id', 'sort_order')
fact_financial_pnl = to_snake_case_df(df_final)'''

In [0]:
df_final = enrich_df(exploded_df)
df_final.display()

In [0]:
df_final = enrich_df2(exploded_df)
df_final.display()

In [0]:
df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master").select("account_name", "sort_order", "calculation_type")
df_final_sort = df_final.join(
        df_coa_master, 
        df_coa_master["account_name"].cast("string") == df_final.account_name,
        'inner'
    ).drop(df_coa_master["account_name"]).select(
        col("account_name"),
        col("name"),
        col("majour_group"),
        col("group"),
        col("sub_group"),
        # lit(None).cast("string"),
        col("accoun_type"),
        col("netsuite_location_name"),
        col("type"),
        col("location_id"),
        col("brand_id"),
        col("company_id"),
        # lit(None).cast("string").alias("Cluster"),
        col("store_type"),
        col("parent_company"),
        col("country_code"),
        col('zone'),
        col("city"),
        col("amount"),
        col("year"),
        col("month"),
        col("location_actual_net_sales"),
        col("location_py_net_sales"),
        col("brand_act_net_sales"),
        col("brand_py_net_sales"),
        col("company_act_net_sales"),
        col("company_py_net_sales"),
        col("detail/total"),
        col('sort_order'),
        col('calculation_type')
    )
df_final_ = df_final_sort.orderBy("parent_company", "company_id", "brand_id", "netsuite_location_name", 'year', 'month', 'sort_order').display()

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_catalog.silver.fact_financial_pnl (
  account_name STRING,
  account_number STRING,
  majour_group STRING,
  group_name STRING,
  sub_group STRING,
  alternate_group STRING,
  account_type STRING,
  location STRING,
  type STRING COMMENT 'HO/Store',
  location_code STRING,
  brand STRING,
  company STRING,
  cluster STRING,
  location_mode STRING COMMENT 'Mall/Drive thru/Stand alone',
  emirates STRING,
  detail_total_grand_total STRING,
  sum_order INT,
  actual_value DECIMAL(18,2),
  budget DECIMAL(18,2),
  previous_year_sales DECIMAL(18,2),
  year INT,
  month INT,
  forecast DECIMAL(18,2),
  calculation_type STRING,
  actual_net_sales DECIMAL(18,2),
  budget_net_sales DECIMAL(18,2),
  py_net_sales DECIMAL(18,2),
  brand_act_net_sales DECIMAL(18,2),
  brand_budget_net_sales DECIMAL(18,2),
  brand_py_net_sales DECIMAL(18,2),
  company_act_net_sales DECIMAL(18,2),
  company_budget_net_sales DECIMAL(18,2),
  company_py_net_sales DECIMAL(18,2)
)
USING DELTA
CLUSTER BY (year, month, company, brand)
LOCATION 'abfss://fq-dev-silver-container@fqadfstoragedev.dfs.core.windows.net/external/fact_financial_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
def merge_stream_fact_financial_pnl(df, i):
    try:
        exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
        
        fact_financial_pnl_upsert = enrich_json(exploded_df)
        fact_financial_pnl_upsert.createOrReplaceTempView("fact_financial_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
            USING (
                SELECT *
                FROM fact_financial_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.location = source.location
                AND target.account_name = source.account_name
                AND target.sum_order = source.sum_order
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print(f"Successfully merged batch {i}")

    # df.sparkSession.sql("""
    #     MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
    #     USING (
    #         SELECT *
    #         FROM (
    #             SELECT *, 
    #                 ROW_NUMBER() OVER (
    #                     PARTITION BY year, month, location, account_name, sum_order
    #                     ORDER BY year DESC  -- or add a load_time column
    #                 ) as rank
    #             FROM fact_financial_pnl_upsert_microbatch
    #         )
    #         WHERE rank = 1
    #     ) as source
    #     ON target.year = source.year
    #         AND target.month = source.month
    #         AND target.location = source.location
    #         AND target.account_name = source.account_name
    #         AND target.sum_order = source.sum_order
    #     WHEN MATCHED THEN UPDATE SET *
    #     WHEN NOT MATCHED THEN INSERT *
    # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_fact_financial_pnl/schema_fact_financial_pnl')
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_fact_financial_pnl)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_fact_financial_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.fact_financial_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.fact_financial_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()